In [2]:
from utils import get_dataset_lines

# Distances Between Leaves Problem
In this chapter, we define the length of a path in a tree as the sum of the lengths of its edges (rather than the number of edges on the path). As a result, the evolutionary distance between two present-day species corresponding to leaves $i$ and $j$ in a tree $T$ is equal to the length of the unique path connecting $i$ and $j$, denoted $d_{i,j}(T)$.

**Code Challenge**: Solve the Distances Between Leaves Problem.

**Input**: An integer $n$ followed by the adjacency list of a weighted tree with $n$ leaves. The tree is given as an adjacency list of a graph whose leaves are integers between $0$ and $n - 1$; the notation `a->b:c` means that node $a$ is connected to node $b$ by an edge of weight $c$.

**Output**: An $n \times n$ matrix $(d_{i,j})$, where $d_{i,j}$ is the length of the path between leaves $i$ and $j$. The matrix you return should be space-separated.

**Sample Input**:

```
4
0->4:11
1->4:2
2->5:6
3->5:7
4->0:11
4->1:2
4->5:4
5->4:4
5->3:7
5->2:6
```

**Sample Output**:

```
0	13	21	22
13	0	12	13
21	12	0	13
22	13	13	0
```

In [3]:
def DistancesBetweenLeaves(n, adjacency_list):
    adj = {}
    for line in adjacency_list:
        line = line.strip()
        if not line: continue
        parts = line.split('->')
        u = int(parts[0])
        v_w = parts[1].split(':')
        v = int(v_w[0])
        w = int(v_w[1])
        
        if u not in adj: adj[u] = []
        adj[u].append((v, w))
        
    dist_matrix = [[0] * n for _ in range(n)]
    
    for i in range(n):
        # Run BFS from leaf i
        visited = {i}
        queue = [(i, 0)] # current_node, current_dist
        
        idx = 0
        while idx < len(queue):
            u, d = queue[idx]
            idx += 1
            
            if u < n:
                dist_matrix[i][u] = d
            
            if u in adj:
                for v, w in adj[u]:
                    if v not in visited:
                        visited.add(v)
                        queue.append((v, d + w))
                        
    return dist_matrix

In [4]:
# Sample Input
n = 4
adjacency_list = [
    "0->4:11",
    "1->4:2",
    "2->5:6",
    "3->5:7",
    "4->0:11",
    "4->1:2",
    "4->5:4",
    "5->4:4",
    "5->3:7",
    "5->2:6"
]

# Run the function
dist_matrix = DistancesBetweenLeaves(n, adjacency_list)

# Print the result
for row in dist_matrix:
    print(" ".join(map(str, row)))

# Test Assertion
expected_output = [
    [0, 13, 21, 22],
    [13, 0, 12, 13],
    [21, 12, 0, 13],
    [22, 13, 13, 0]
]
assert dist_matrix == expected_output, f"Expected {expected_output}, but got {dist_matrix}"
print("Test passed!")

0 13 21 22
13 0 12 13
21 12 0 13
22 13 13 0
Test passed!


In [5]:
# Test Dataset
test_dataset_filename = 'dataset_30284_12.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    n = int(lines[0])
    adjacency_list = lines[1:]
    
    dist_matrix = DistancesBetweenLeaves(n, adjacency_list)
    for row in dist_matrix:
        print(" ".join(map(str, row)))
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

0 80 33 132 114 142 180 110 76 131 82 180 100 48 126 65 145 18 30 125 107 96 165 70 25 160 60 58 56 147 44 33
80 0 103 202 60 212 126 180 146 77 152 126 170 118 72 37 91 88 64 195 53 42 111 140 81 106 130 50 126 217 56 103
33 103 0 125 137 135 203 103 69 154 75 203 93 41 149 88 168 27 53 118 130 119 188 63 48 183 53 81 49 140 67 26
132 202 125 0 236 34 302 48 66 253 60 302 58 114 248 187 267 126 152 33 229 218 287 78 147 282 82 180 96 39 166 113
114 60 137 236 0 246 86 214 180 37 186 86 204 152 32 71 51 122 98 229 35 48 71 174 115 66 164 84 160 251 90 137
142 212 135 34 246 0 312 58 76 263 70 312 68 124 258 197 277 136 162 43 239 228 297 88 157 292 92 190 106 19 176 123
180 126 203 302 86 312 0 280 246 63 252 26 270 218 82 137 51 188 164 295 101 114 27 240 181 38 230 150 226 317 156 203
110 180 103 48 214 58 280 0 44 231 38 280 36 92 226 165 245 104 130 41 207 196 265 56 125 260 60 158 74 63 144 91
76 146 69 66 180 76 246 44 0 197 16 246 34 58 192 131 211 70 96 59 173 162 231 22 91 226

# Limb Length Problem
We now have an algorithm for solving the Limb Length Problem. For each $j$, we can compute $LimbLength(j)$ by finding the minimum value of $(D_{i,j} + D_{j,k} - D_{i,k})/2$ over all pairs of leaves $i$ and $k$ (where $i \neq j$ and $k \neq j$).

**Code Challenge**: Solve the Limb Length Problem.

**Input**: An integer $n$, followed by an integer $j$ between $0$ and $n - 1$, followed by a space-separated additive distance matrix $D$ (whose elements are integers).

**Output**: The limb length of the leaf in $Tree(D)$ corresponding to row $j$ of this distance matrix (use 0-based indexing).

**Sample Input**:

```
4
1
0	13	21	22
13	0	12	13
21	12	0	13
22	13	13	0
```

**Sample Output**:

```
2
```

In [6]:
def LimbLength(n, j, D):
    min_length = float('inf')
    
    for i in range(n):
        for k in range(n):
            if i != j and k != j:
                current_length = (D[i][j] + D[j][k] - D[i][k]) // 2
                if current_length < min_length:
                    min_length = current_length
                    
    return min_length

In [7]:
# Sample Input
n = 4
j = 1
D = [
    [0, 13, 21, 22],
    [13, 0, 12, 13],
    [21, 12, 0, 13],
    [22, 13, 13, 0]
]

# Run the function
result = LimbLength(n, j, D)
print(result)

# Test Assertion
expected_output = 2
assert result == expected_output, f"Expected {expected_output}, but got {result}"
print("Test passed!")

2
Test passed!


In [8]:
# Test Dataset
test_dataset_filename = 'dataset_30285_11.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    n = int(lines[0])
    j = int(lines[1])
    D = []
    for line in lines[2:]:
        D.append(list(map(int, line.split())))
        
    print(LimbLength(n, j, D))
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

322


# Distance-Based Phylogeny Problem
**Code Challenge**: Implement AdditivePhylogeny to solve the Distance-Based Phylogeny Problem.

**Input**: An integer $n$ followed by a space-separated $n \times n$ distance matrix.

**Output**: A weighted adjacency list for the simple tree fitting this matrix.

**Note on formatting**: The adjacency list must have consecutive integer node labels starting from 0. The $n$ leaves must be labeled $0, 1, \dots, n - 1$ in order of their appearance in the distance matrix. Labels for internal nodes may be labeled in any order but must start from $n$ and increase consecutively.

**Sample Input**:

```
4
0	13	21	22
13	0	12	13
21	12	0	13
22	13	13	0
```

**Sample Output**:

```
0->4:11
1->4:2
2->5:6
3->5:7
4->0:11
4->1:2
4->5:4
5->4:4
5->3:7
5->2:6
```

In [9]:
def find_path(tree, start, end):
    queue = [(start, [start])]
    visited = {start}
    
    while queue:
        curr, path = queue.pop(0)
        if curr == end:
            return path
        
        if curr in tree:
            for neighbor, weight in tree[curr]:
                if neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, path + [neighbor]))
    return None

def get_edge_weight(tree, u, v):
    for neighbor, weight in tree[u]:
        if neighbor == v:
            return weight
    return 0

def AdditivePhylogeny(D, n, counter):
    if n == 2:
        T = {}
        w = D[0][1]
        T[0] = [(1, w)]
        T[1] = [(0, w)]
        return T
    
    limb_len = LimbLength(n, n-1, D)
    
    # Modify D in place for the recursive step logic
    # We need to be careful not to affect the outer scope if D is reused, 
    # but here D is passed down and we only care about the current step.
    # However, since we slice D to get D_prime, we should modify D first.
    
    for j in range(n - 1):
        D[j][n-1] -= limb_len
        D[n-1][j] = D[j][n-1]
        
    i_found, k_found = -1, -1
    for i in range(n - 1):
        for k in range(n - 1):
            if i != k:
                if D[i][k] == D[i][n-1] + D[n-1][k]:
                    i_found, k_found = i, k
                    break
        if i_found != -1: break
        
    x = D[i_found][n-1]
    
    # Create D_prime (n-1 x n-1)
    D_prime = [row[:n-1] for row in D[:n-1]]
    
    T = AdditivePhylogeny(D_prime, n - 1, counter)
    
    # Add leaf n-1 back
    path = find_path(T, i_found, k_found)
    
    curr_dist = 0
    v = -1
    
    # Traverse path to find attachment point
    for idx in range(len(path) - 1):
        u = path[idx]
        w = path[idx+1]
        weight = get_edge_weight(T, u, w)
        
        if curr_dist + weight > x:
            # v is on edge (u, w)
            dist_u_v = x - curr_dist
            dist_v_w = weight - dist_u_v
            
            v = counter[0]
            counter[0] += 1
            
            # Remove edge (u, w)
            T[u].remove((w, weight))
            T[w].remove((u, weight))
            
            # Add edges (u, v), (v, w)
            if u not in T: T[u] = []
            T[u].append((v, dist_u_v))
            
            if w not in T: T[w] = []
            T[w].append((v, dist_v_w))
            
            if v not in T: T[v] = []
            T[v].append((u, dist_u_v))
            T[v].append((w, dist_v_w))
            
            break
            
        elif curr_dist + weight == x:
            v = w
            break
            
        curr_dist += weight
        
    if v == -1 and x == 0:
        v = i_found
        
    # Add leaf n-1 attached to v
    leaf_node = n - 1
    if leaf_node not in T: T[leaf_node] = []
    T[leaf_node].append((v, limb_len))
    
    if v not in T: T[v] = []
    T[v].append((leaf_node, limb_len))
    
    return T

def SolveAdditivePhylogeny(n, D):
    # Make a deep copy of D because AdditivePhylogeny modifies it
    D_copy = [row[:] for row in D]
    counter = [n] # Mutable integer for internal node IDs
    T = AdditivePhylogeny(D_copy, n, counter)
    return T

In [10]:
# Sample Input
n = 4
D = [
    [0, 13, 21, 22],
    [13, 0, 12, 13],
    [21, 12, 0, 13],
    [22, 13, 13, 0]
]

# Run the function
T = SolveAdditivePhylogeny(n, D)

# Print the result
sorted_nodes = sorted(T.keys())
for u in sorted_nodes:
    neighbors = T[u]
    sorted_neighbors = sorted(neighbors, key=lambda x: x[0])
    for v, w in sorted_neighbors:
        print(f"{u}->{v}:{w}")

# Test Assertion (checking specific edges from sample output)
expected_edges = {
    0: [(4, 11)],
    1: [(4, 2)],
    2: [(5, 6)],
    3: [(5, 7)],
    4: [(0, 11), (1, 2), (5, 4)],
    5: [(4, 4), (3, 7), (2, 6)]
}

# Helper to check if edges match
def check_edges(T, expected):
    for u, edges in expected.items():
        if u not in T: return False
        t_edges = sorted(T[u])
        e_edges = sorted(edges)
        if t_edges != e_edges: return False
    return True

assert check_edges(T, expected_edges), "Tree structure does not match expected output"
print("Test passed!")

0->4:11
1->4:2
2->5:6
3->5:7
4->0:11
4->1:2
4->5:4
5->2:6
5->3:7
5->4:4
Test passed!


In [11]:
# Test Dataset
test_dataset_filename = 'dataset_30286_6.txt'
try:
    lines = get_dataset_lines(test_dataset_filename)
    n = int(lines[0])
    D = []
    for line in lines[1:]:
        D.append(list(map(int, line.split())))
        
    T = SolveAdditivePhylogeny(n, D)
    
    sorted_nodes = sorted(T.keys())
    for u in sorted_nodes:
        neighbors = T[u]
        sorted_neighbors = sorted(neighbors, key=lambda x: x[0])
        for v, w in sorted_neighbors:
            print(f"{u}->{v}:{w}")
            
except FileNotFoundError:
    print(f"File {test_dataset_filename} not found. Please download the dataset and check the filename.")
except Exception as e:
    print(f"An error occurred: {e}")

0->23:228
1->21:708
2->24:378
3->22:508
4->25:61
5->24:221
6->25:467
7->26:235
8->27:679
9->28:514
10->29:778
11->30:160
12->31:90
13->32:395
14->33:203
15->34:703
16->35:432
17->36:233
18->37:889
19->38:702
20->39:861
21->1:708
21->38:944
21->39:91
22->3:508
22->34:751
22->37:455
23->0:228
23->25:156
23->27:248
24->2:378
24->5:221
24->35:118
25->4:61
25->6:467
25->23:156
26->7:235
26->28:478
26->32:879
27->8:679
27->23:248
27->36:528
28->9:514
28->26:478
28->31:417
29->10:778
29->30:759
29->35:650
30->11:160
30->29:759
30->38:785
31->12:90
31->28:417
31->36:455
32->13:395
32->26:879
32->33:645
33->14:203
33->32:645
33->34:335
34->15:703
34->22:751
34->33:335
35->16:432
35->24:118
35->29:650
36->17:233
36->27:528
36->31:455
37->18:889
37->22:455
37->39:113
38->19:702
38->21:944
38->30:785
39->20:861
39->21:91
39->37:113


In [ ]:
# Coursera Quiz Questions

# Q1: Compute LimbLength(i) for the additive distance matrix shown below.
D = [
    [0, 14, 17, 17],
    [14, 0, 7, 13],
    [17, 7, 0, 16],
    [17, 13, 16, 0]
]
print(LimbLength(4, 0, D))
print((D[0][1]+D[1][2]-D[0][2]) // 2)

9
17
7
0
16
